In [1]:
from pathlib import Path
import os
from jaiai.etc.io import source_from_dataset
import json
import dotenv
import nest_asyncio
from collections import Counter
import json_repair
from loguru import logger
import polars as pl

nest_asyncio.apply()
dotenv.load_dotenv()

True

###
<p> Укажите путь до вашего датасета </p>
<code>fpath = ... </code>

Обязательные поля:
 - `text`

In [ ]:
fpath = Path(os.getcwd()) / "data" / "gazprombank.10k.dataset.json"

In [ ]:
js_docs = source_from_dataset(fpath, as_json=True)

In [3]:
js_docs[0]

{'id': 11302079,
 'date': 1703520000,
 'text': 'Хотел бы поделиться своим впечатлением от использования премиальной карты Газпромбанка Mir Supreme. При посещении офиса Газпромбанка в г. Оренбург меня убедили оформить данную премиальную карту. Не видел никаких преимуществ этой карты для себя, но всё же оформил её надеясь на высокий стандарт обслуживания (https://www.gazprombank.ru/press/6974393/) и преимущества во вкладках/накопительных счетах. При оформлении карты мне сказали, что я могу, при посещении офиса банка (предварительно показав свою карту Mir Supreme), обратиться к персональному менеджеру по любому вопросу, также можно выбрать одну из программ «Спорт» с бесплатным фитнес-абонементом или «Комфортное путешествие». \n\nЧерез несколько месяцев мне нужно было посетить офис банка и, заодно, посмотреть и почувствовать, что значит высокий стандарт обслуживания. При входе в офис банка я показал свою премиальную карту сотруднику банка с просьбой предоставить мне возможность поговорить 

In [4]:
from jaiai.running.instructions import InstructionForTopicExtraction, OpenAiTask

/home/justatom/miniconda3/envs/jaiai/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
pipeline = InstructionForTopicExtraction(
    system_prompt_fpath=Path(os.getcwd()) / "jaiai" / "builtins" / "entities_system_prompt.txt"
)

In [6]:
pipeline.system_prompt

'Ты — модель для извлечения тематик и тональностей из русскоязычных банковских отзывов (Financial Sector).\nНельзя упоминать бренд банка (все отзывы с одного банка).\nВывод строго в 2 строки markdown:\ntopics: ["тема_1", "тема_2", ...]\nsentiments: ["положительно"|"отрицательно"|"нейтрально", ...]\nДлины списков совпадают, 1–5 тем, только нижний регистр тональностей.\nТаксономия (нормализуй к ней, без брендов/имен/дат):\n— Карты: оформление карты, перевыпуск карты, выдача карты, блокировка карты, операции по карте, комиссии по карте, лимиты по карте, кэшбэк\n— Счета/вклады: текущий счет, накопительный счет, вклад, процентная ставка, пополнение счета, списание со счета, заморозка счета\n— Дистанционные каналы: мобильное приложение, интернет-банк, сайт банка, уведомления\n— Качество сервиса: обслуживание, компетентность специалиста, время ожидания, вежливость персонала, поддержка/контакт-центр\n— Инфраструктура: офис банка, электронная очередь, банкомат, терминал\n— Надзор/комплаенс: иде

In [7]:
tasks = [OpenAiTask(id=js_doc["id"], text=js_doc["text"]) for js_doc in js_docs]

In [10]:
response = await pipeline.run_async(tasks)

2025-10-02 03:40:49.410 | INFO     | jaiai.running.mask:run_async:73 - run_async: starting 10000 tasks
2025-10-02 03:40:50.297 | INFO     | jaiai.running.mask:logger_callback:79 - OpenAIAsyncWrapper: completed 1/10000 tasks (0.01%)
2025-10-02 03:40:51.255 | INFO     | jaiai.running.mask:logger_callback:79 - OpenAIAsyncWrapper: completed 2/10000 tasks (0.02%)
2025-10-02 03:40:52.153 | INFO     | jaiai.running.mask:logger_callback:79 - OpenAIAsyncWrapper: completed 3/10000 tasks (0.03%)
2025-10-02 03:40:53.197 | INFO     | jaiai.running.mask:logger_callback:79 - OpenAIAsyncWrapper: completed 4/10000 tasks (0.04%)
2025-10-02 03:40:53.819 | INFO     | jaiai.running.mask:logger_callback:79 - OpenAIAsyncWrapper: completed 5/10000 tasks (0.05%)
2025-10-02 03:40:54.578 | INFO     | jaiai.running.mask:logger_callback:79 - OpenAIAsyncWrapper: completed 6/10000 tasks (0.06%)
2025-10-02 03:40:55.672 | INFO     | jaiai.running.mask:logger_callback:79 - OpenAIAsyncWrapper: completed 7/10000 tasks (0

In [11]:
response[0]

{'id': 11781687,
 'metadata': {},
 'model': 'llama-3.1-8b-instruct',
 'error': None,
 'text': 'topics: ["дистанционные каналы", "качество сервиса"]\nsentiments: ["отрицательно", "отрицательно"]',
 'parsed_json': {'topics': ['дистанционные каналы', 'качество сервиса'],
  'sentiments': ['отрицательно', 'отрицательно']},
 'raw': {'id': 'chatcmpl-vWs3nD6oA8oFzJ8NXn5FzaFLLnxwmmvQ',
  'choices': [{'finish_reason': 'stop',
    'index': 0,
    'logprobs': None,
    'message': {'content': 'topics: ["дистанционные каналы", "качество сервиса"]\nsentiments: ["отрицательно", "отрицательно"]',
     'refusal': None,
     'role': 'assistant',
     'annotations': None,
     'audio': None,
     'function_call': None,
     'tool_calls': None}}],
  'created': 1759365650,
  'model': 'llama-3.1-8b-instruct',
  'object': 'chat.completion',
  'service_tier': None,
  'system_fingerprint': 'b5747-0142961a',
  'usage': {'completion_tokens': 35,
   'prompt_tokens': 760,
   'total_tokens': 795,
   'completion_toke

In [12]:
js_docs[0]

{'id': 11302079,
 'date': 1703520000,
 'text': 'Хотел бы поделиться своим впечатлением от использования премиальной карты Газпромбанка Mir Supreme. При посещении офиса Газпромбанка в г. Оренбург меня убедили оформить данную премиальную карту. Не видел никаких преимуществ этой карты для себя, но всё же оформил её надеясь на высокий стандарт обслуживания (https://www.gazprombank.ru/press/6974393/) и преимущества во вкладках/накопительных счетах. При оформлении карты мне сказали, что я могу, при посещении офиса банка (предварительно показав свою карту Mir Supreme), обратиться к персональному менеджеру по любому вопросу, также можно выбрать одну из программ «Спорт» с бесплатным фитнес-абонементом или «Комфортное путешествие». \n\nЧерез несколько месяцев мне нужно было посетить офис банка и, заодно, посмотреть и почувствовать, что значит высокий стандарт обслуживания. При входе в офис банка я показал свою премиальную карту сотруднику банка с просьбой предоставить мне возможность поговорить 

In [14]:
response[0]

{'id': 11781687,
 'metadata': {},
 'model': 'llama-3.1-8b-instruct',
 'error': None,
 'text': 'topics: ["дистанционные каналы", "качество сервиса"]\nsentiments: ["отрицательно", "отрицательно"]',
 'parsed_json': {'topics': ['дистанционные каналы', 'качество сервиса'],
  'sentiments': ['отрицательно', 'отрицательно']},
 'raw': {'id': 'chatcmpl-vWs3nD6oA8oFzJ8NXn5FzaFLLnxwmmvQ',
  'choices': [{'finish_reason': 'stop',
    'index': 0,
    'logprobs': None,
    'message': {'content': 'topics: ["дистанционные каналы", "качество сервиса"]\nsentiments: ["отрицательно", "отрицательно"]',
     'refusal': None,
     'role': 'assistant',
     'annotations': None,
     'audio': None,
     'function_call': None,
     'tool_calls': None}}],
  'created': 1759365650,
  'model': 'llama-3.1-8b-instruct',
  'object': 'chat.completion',
  'service_tier': None,
  'system_fingerprint': 'b5747-0142961a',
  'usage': {'completion_tokens': 35,
   'prompt_tokens': 760,
   'total_tokens': 795,
   'completion_toke

In [15]:
def formatted_response(raw: dict):
    raw_answer = raw["text"]
    parsed_json = raw["parsed_json"]
    id_message = raw["id"]
    topics, sentiments = [], []
    if parsed_json is not None:
        topics, sentiments = parsed_json["topics"], parsed_json["sentiments"]
    else:
        js_res = json_repair.loads(raw_answer)
        if isinstance(js_res, list) and len(js_res) >= 2:
            topics: list[str] = js_res[0]
            sentiments: list[str] = js_res[1]
            if len(sentiments) > len(topics):
                _counter = Counter(sentiments)
                top_sentiment_per_topic = _counter.most_common(1)[0][0]
                sentiments.extend([top_sentiment_per_topic] * (len(sentiments) - len(topics)))
            elif len(sentiments) < len(topics):
                topics = topics[: len(sentiments)]
        else:
            logger.warning(
                f"For response {raw_answer} neither `raw_json` is present and unable to hand-craft JSON compatable answer. Parsed json is not a list but {type(js_res)}"
            )
    return dict(topics=topics, sentiments=sentiments, id=id_message)

In [19]:
len(response)

10000

In [ ]:
import gc

gc.collect()

2371

In [22]:
llm_response_present = [res for res in response if res["parsed_json"] is not None]

In [23]:
formatted_llm_response = [formatted_response(res) for res in llm_response_present]

In [24]:
pl_docs = pl.from_dicts(js_docs)
pl_responses = pl.from_dicts(formatted_llm_response)

In [25]:
pl_final_response = pl_docs.join(pl_responses, on="id", how="inner")

In [28]:
pl_final_response.shape

(6180, 6)

In [27]:
pl_final_response.head()

id,date,text,bank,topics,sentiments
i64,i64,str,str,list[str],list[str]
11302079,1703520000,"""Хотел бы подел…","""Газпромбанк""","[""оформление карты"", ""ожидание в офисе"", … ""непонятные списания""]","[""отрицательно"", ""отрицательно"", … ""отрицательно""]"
11401323,1710781020,"""Решила оформит…","""Газпромбанк""","[""оформление именной карты"", ""коммуникация с банковскими специалистами"", ""условия использования именной карты""]","[""отрицательно"", ""отрицательно"", ""отрицательно""]"
11784070,1728589680,"""21.09.2024 год…","""Газпромбанк""","[""заблокированные счета"", ""перевод средств"", … ""проблемы с накопительным счетом""]","[""отрицательно"", ""отрицательно"", … ""отрицательно""]"
11761378,1727855640,"""Добрый день! …","""Газпромбанк""","[""распоряжение персональными данными"", ""ожидание ответа банка"", ""качество поддержки""]","[""отрицательно"", ""отрицательно"", ""отрицательно""]"
11353808,1706901060,"""Материнский ка…","""Газпромбанк""","[""материнский капитал""]","[""отрицательно""]"


In [29]:
pl_final_response.write_excel(f"gazprom.dataset.6.2k.topics.xlsx")

In [30]:
with open(f"gazprom.dataset.6.2k.topics.json", "w+") as fp:
    json.dump(pl_final_response.to_dicts(), fp, ensure_ascii=False)

In [ ]:
problematic_responses = [res for res in response if res["parsed_json"] == None]

In [32]:
json_repair.loads(problematic_responses[12]["raw"]["choices"][0]["message"]["content"])

[['операции по карте',
  'комиссии по карте',
  'дистанционные каналы',
  'качество сервиса'],
 ['отрицательно', 'отрицательно', 'отрицательно']]

In [ ]:
Counter(["отрицательно", "отрицательно", "отрицательно", "положительно", "положительно", "положительно"]).most_common(1)[0]

('отрицательно', 3)

In [51]:
def formatted_response(raw: dict):
    raw_answer = raw["text"]
    parsed_json = raw["parsed_json"]
    id_message = raw["id"]
    topics, sentiments = [], []
    if parsed_json is not None:
        topics, sentiments = parsed_json["topics"], parsed_json["sentiments"]
    else:
        js_res = json_repair.loads(raw_answer)
        if isinstance(js_res, list) and len(js_res) >= 2:
            topics: list[str] = js_res[0]
            sentiments: list[str] = js_res[1]
            if len(sentiments) > len(topics):
                _counter = Counter(sentiments)
                top_sentiment_per_topic = _counter.most_common(1)[0][0]
                sentiments.extend([top_sentiment_per_topic] * (len(sentiments) - len(topics)))
            elif len(sentiments) < len(topics):
                topics = topics[: len(sentiments)]
        else:
            logger.warning(
                f"For response {raw_answer} neither `raw_json` is present and unable to hand-craft JSON compatable answer. Parsed json is not a list but {type(js_res)}"
            )
    return dict(topics=topics, sentiments=sentiments, id=id_message)

In [52]:
[formatted_response(res) for res in problematic_responses]

[{'topics': ['оформление карты', 'комиссии по карте', 'дистанционные каналы'],
  'sentiments': ['отрицательно', 'отрицательно', 'отрицательно'],
  'id': 11591025},
 {'topics': ['качество сервиса', 'дистанционные каналы', 'платежи/переводы'],
  'sentiments': ['отрицательно', 'отрицательно', 'отрицательно'],
  'id': 11794862},
 {'topics': ['переводы', 'комиссии', 'кредитная карта'],
  'sentiments': ['отрицательно', 'отрицательно', 'отрицательно'],
  'id': 11723768},
 {'topics': ['консультация'], 'sentiments': ['положительно'], 'id': 12355239},
 {'topics': ['качество сервиса', 'дистанционные каналы', 'тарифы/условия'],
  'sentiments': ['положительно', 'положительно', 'положительно'],
  'id': 11351835},
 {'topics': ['качество сервиса', 'дистанционные каналы', 'платежи/переводы'],
  'sentiments': ['отрицательно', 'отрицательно', 'отрицательно'],
  'id': 11683187},
 {'topics': ['дебетовая карта', 'бонусы', 'горячая линия'],
  'sentiments': ['отрицательно', 'отрицательно', 'отрицательно'],
  

In [54]:
formatted_response(response[0])

{'topics': ['блокировка карты', 'возврат денег'],
 'sentiments': ['отрицательно', 'отрицательно'],
 'id': 11644876}

In [ ]:
for res in problematic_responses:
    # json_repair.loads(response[1]["raw"]["choices"][0]["message"]["content"])
    json_repair.loads(res["raw"]["choices"][0]["message"]["content"])

In [26]:
problematic_responses

[{'id': 11591025,
  'metadata': {},
  'model': 'llama-3.1-8b-instruct',
  'error': None,
  'text': 'topics: ["оформление карты", "комиссии по карте", "дистанционные каналы", "качество сервиса"]\nsentiments: ["отрицательно", "отрицательно", "отрицательно"]',
  'parsed_json': None,
  'raw': {'id': 'chatcmpl-NC9r6avC53jQUggoh5DMIGICjdpC3RUJ',
   'choices': [{'finish_reason': 'stop',
     'index': 0,
     'logprobs': None,
     'message': {'content': 'topics: ["оформление карты", "комиссии по карте", "дистанционные каналы", "качество сервиса"]\nsentiments: ["отрицательно", "отрицательно", "отрицательно"]',
      'refusal': None,
      'role': 'assistant',
      'annotations': None,
      'audio': None,
      'function_call': None,
      'tool_calls': None}}],
   'created': 1759311308,
   'model': 'llama-3.1-8b-instruct',
   'object': 'chat.completion',
   'service_tier': None,
   'system_fingerprint': 'b5747-0142961a',
   'usage': {'completion_tokens': 56,
    'prompt_tokens': 719,
    'to

In [47]:
import json_repair

In [1]:
response[1]

NameError: name 'response' is not defined

In [ ]:
json_repair.loads(response[1]["raw"]["choices"][0]["message"]["content"])

[['блокировка карты',
  'перевод',
  'оформление карты',
  'комиссии по карте',
  'лимит по карте',
  'кассир',
  'ошибка оператора',
  'время ожидания',
  'расстояние до офиса',
  'необходимость в машине',
  'заявление',
  'время разблокировки',
  'необходимость в дополнительной карте',
  'недостаток средств'],
 ['отрицательно',
  'отрицательно',
  'отрицательно',
  'отрицательно',
  'отрицательно',
  'отрицательно',
  'отрицательно',
  'отрицательно',
  'отрицательно',
  'отрицательно',
  'отрицательно',
  'отрицательно',
  'отрицательно']]

In [ ]:
import time, datetime

ts = int(time.mktime(datetime.datetime.today().timetuple()))

In [14]:
ts

1759177228

In [13]:
# may be in milliseconds, try `ts /= 1000` in that case
print(datetime.datetime.fromtimestamp(ts).strftime("%Y-%m-%d %H:%M:%S"))

2025-09-29 23:20:28


In [ ]:
import random
import time
import uuid


def generate_sample(n: int) -> dict:
    positives = [random.randint(0, 200) for _ in range(n)]
    negatives = [random.randint(0, 200) for _ in range(n)]
    neutrals = [random.randint(0, 500) for _ in range(n)]
    # Генерируем последовательные даты в виде timestamp (секунды)
    now = int(time.time())
    dates = [str(now - i * 86400) for i in range(n)]  # последние n дней
    sample = {
        "positives": positives,
        "negatives": negatives,
        "neutrals": neutrals,
        "dates": dates,
        "uuid": str(uuid.uuid4()),
    }
    return sample


# Пример использования:
samples = generate_sample(1024)

In [27]:
samples["positives"][0]

184

In [ ]:
samples[""]

In [24]:
with open(f"POST.ts_render.response.json", "w+") as fp:
    json.dump(samples, fp, ensure_ascii=False)